In [1]:
import numpy as np
import math
import pandas as pd
from scipy.stats import norm
from scipy.stats import multivariate_normal
from numba import njit

In [2]:
def group_sequential_designs_recursive(
        n_analyses=3,
        upper_bounds=[2.5, 2, 1.5],
        lower_bounds=[0, 0.75, 1.5],
        n_patients=20,
        null_hypothesis=0,
        alt_hypothesis=0.5,
        variance=1,
        grid_points=75): # Number of grid points for 1D numerical integration

    upper_bounds = np.asarray(upper_bounds, dtype=float)
    lower_bounds = np.asarray(lower_bounds, dtype=float)
    n_patients_analysis = n_patients * np.arange(1, n_analyses + 1)

    # Scale and expected mean increments between analyses
    scale = np.sqrt(n_patients_analysis / (2 * variance))
    mean_0 = null_hypothesis * scale
    mean_1 = alt_hypothesis * scale

    # Pre-allocate output arrays
    futility_null = np.zeros(n_analyses)
    efficacy_null = np.zeros(n_analyses)
    futility_alt = np.zeros(n_analyses)
    efficacy_alt = np.zeros(n_analyses)

    # Define a helper function to run the recursion for a given mean vector
    def run_recursion(means):
        fut_probs = np.zeros(n_analyses)
        eff_probs = np.zeros(n_analyses)
        
        # --- Analysis 1 (Standard Normal Base Case) ---
        # Under analysis 1, Z1 ~ N(means[0], 1)
        fut_probs[0] = norm.cdf(lower_bounds[0], loc=means[0], scale=1)
        eff_probs[0] = 1.0 - norm.cdf(upper_bounds[0], loc=means[0], scale=1)
        
        # Create numerical grid for the continuation region at analysis 1
        # If the continuation region is invalid/closed, we stop early
        if lower_bounds[0] >= upper_bounds[0]:
            return fut_probs, eff_probs
            
        z_grid = np.linspace(lower_bounds[0], upper_bounds[0], grid_points)
        # Numerical density distribution of staying in the trial at stage 1
        density = norm.pdf(z_grid, loc=means[0], scale=1)
        
        # --- Analyses 2 through N ---
        for k in range(1, n_analyses):
            # Compute correlation parameters between stage k and k+1
            I_prev = n_patients_analysis[k-1]
            I_curr = n_patients_analysis[k]
            
            # Independent increment properties
            rho = np.sqrt(I_prev / I_curr) 
            cond_scale = np.sqrt(1 - rho**2) # Conditional standard deviation
            
            # Step 1: Compute cumulative probabilities for stopping at stage k+1
            # We integrate: P(Z_curr < bound) = \int [P(Z_curr < bound | Z_prev=z) * density(z)] dz
            # The conditional mean is: means[k] + rho * (z - means[k-1])
            cond_mean_func = lambda z: means[k] + rho * (z - means[k-1])
            
            # Vectorized trapezoidal integration over the previous stage's z_grid
            dz = z_grid[1] - z_grid[0]
            
            # Futility probability at stage k
            cond_fut = norm.cdf(lower_bounds[k], loc=cond_mean_func(z_grid), scale=cond_scale)
            fut_probs[k] = np.trapz(cond_fut * density, dx=dz)
            
            # Efficacy probability at stage k
            cond_eff = 1.0 - norm.cdf(upper_bounds[k], loc=cond_mean_func(z_grid), scale=cond_scale)
            eff_probs[k] = np.trapz(cond_eff * density, dx=dz)
            
            # Step 2: Propagate the density grid forward to the NEXT stage's continuation region
            if k < n_analyses - 1:
                next_lower, next_upper = lower_bounds[k], upper_bounds[k]
                if next_lower >= next_upper:
                    break
                
                next_z_grid = np.linspace(next_lower, next_upper, grid_points)
                
                # Transition matrix computation via outer broadcasting
                # Rows = next stage grid points, Columns = current stage grid points
                c_means = cond_mean_func(z_grid) # shape (grid_points,)
                transition_pdf = norm.pdf(next_z_grid[:, None], loc=c_means, scale=cond_scale)
                
                # Matrix multiply transition probabilities with current density profile
                density = np.trapz(transition_pdf * density, dx=dz, axis=1)
                z_grid = next_z_grid
                
        return fut_probs, eff_probs

    # Run the 1D recursive engine for both Null and Alternative hypotheses
    futility_null, efficacy_null = run_recursion(mean_0)
    futility_alt, efficacy_alt = run_recursion(mean_1)

    alpha = efficacy_null.sum()
    power = efficacy_alt.sum()
    summed_probs = futility_alt + efficacy_alt
    expected_sample_size = np.sum(summed_probs * n_patients_analysis)

    return alpha, power, expected_sample_size

In [3]:
# simulate the trials to obtain alpha and beta
def group_sequential_designs_mazin(
        n_analyses=3,
        upper_bounds=[2.5, 2, 1.5],
        lower_bounds=[0, 0.75, 1.5],
        n_patients=20,
        null_hypothesis=0,
        alt_hypothesis=0.5,
        variance=1):

    upper_bounds = np.asarray(upper_bounds, dtype=float)
    lower_bounds = np.asarray(lower_bounds, dtype=float)

    # cumulative sample sizes
    n_patients_analysis = n_patients * np.arange(1, n_analyses + 1)

    # compute the means
    scale = np.sqrt(n_patients_analysis / (2 * variance))

    # mean_0 is under null, mean_1 is under alternative
    mean_0 = null_hypothesis * scale
    mean_1 = alt_hypothesis * scale

    # n_i becomes a (x by 1) vector
    # n_j becomes a (1 by x) vector
    n_i = n_patients_analysis[:, None]
    n_j = n_patients_analysis[None, :]

    # np.minimum and np.maximum broadcasts the vectors into (x by x)
    # matrix where np.minimum contains the min across the rows and colums
    # comparing each dimension and np.maximum contains the max across
    # the rows and columns comparing each dimension
    full_sigma = np.sqrt(np.minimum(n_i, n_j) / np.maximum(n_i, n_j))

    # store results in arrays first, then add to DataFrame at the end
    futility_null = np.empty(n_analyses)
    efficacy_null = np.empty(n_analyses)
    futility_alt = np.empty(n_analyses)
    efficacy_alt = np.empty(n_analyses)

    for i in range(n_analyses):

        # the number of dimensions to subselect
        dim = i + 1

        # select a (dim by dim) section of the matrix
        cov = full_sigma[:dim, :dim]

        mvn_null = multivariate_normal(
            mean=mean_0[:dim],
            cov=cov
        )

        mvn_alt = multivariate_normal(
            mean=mean_1[:dim],
            cov=cov
        )

        # Futility bounds
        fut_l = np.concatenate([lower_bounds[:i], [-np.inf]])
        fut_u = np.concatenate([upper_bounds[:i], [lower_bounds[i]]])

        # Efficacy bounds
        eff_l = np.concatenate([lower_bounds[:i], [upper_bounds[i]]])
        eff_u = np.concatenate([upper_bounds[:i], [np.inf]])

        futility_null[i] = mvn_null.cdf(fut_u, lower_limit=fut_l)
        futility_alt[i] = mvn_alt.cdf(fut_u, lower_limit=fut_l)

        efficacy_null[i] = mvn_null.cdf(eff_u, lower_limit=eff_l)
        efficacy_alt[i] = mvn_alt.cdf(eff_u, lower_limit=eff_l)

    # get alpha and power; alpha is the error of claiming efficacy
    # under the null. Power is the "correct" call of efficacy under
    # the alternative
    alpha = efficacy_null.sum()
    power = efficacy_alt.sum()

    # calculate the expected sample size by summing stopping for
    # futility or efficacy under the alternative
    summed_probs = futility_alt + efficacy_alt
    expected_sample_size = np.sum(summed_probs * n_patients_analysis)

    return alpha, power, expected_sample_size


In [4]:
@njit
def _numba_grid_engine(n_analyses, upper_bounds, lower_bounds, n_patients_analysis, means, grid_points):
    fut_probs = np.zeros(n_analyses)
    eff_probs = np.zeros(n_analyses)

    ####################
    # helper functions #
    ####################

    # standard normal pdf (mean=0, std=1)
    def std_norm_pdf(x):
        return np.exp(-0.5 * x**2) / np.sqrt(2.0 * np.pi)
        
    # standard normal cdf (mean=0, std=1)
    def std_norm_cdf(x):
        return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

    ##############
    # analysis 1 #
    ##############
    
    # for the first stage, compute the probabilities of stopping
    # for futility and efficacy; bound - means standardizes the problem
    fut_probs[0] = std_norm_cdf(lower_bounds[0] - means[0])
    eff_probs[0] = 1.0 - std_norm_cdf(upper_bounds[0] - means[0])
    
    if lower_bounds[0] >= upper_bounds[0]:
        print("Error: Upper bounds are lower than lower bounds.")
        return

    # z_grid is selecting a number of grid points and splitting up
    # the continuation region (ell < cont_region < u)
    z_grid = np.linspace(lower_bounds[0], upper_bounds[0], grid_points)

    # fill in the probability densities (PDF NOT CDF!) for each of
    # the selected points in the continuation region; again
    # z_grid - means standardizes
    density = np.zeros(grid_points)
    for i in range(grid_points):
        density[i] = std_norm_pdf(z_grid[i] - means[0])

    ###################
    # analyses 2 to K #
    ###################
    
    for k in range(1, n_analyses):

        I_prev = n_patients_analysis[k-1]
        I_curr = n_patients_analysis[k]

        # this is the covariance between the previous 
        # and current stages Cov(z_prev, z_curr)
        cov_pre_curr = np.sqrt(I_prev / I_curr)

        # the conditional standard deviation
        cond_scale = np.sqrt(1.0 - cov_pre_curr**2)

        # the width of the steps along the grid as calculated above
        dz = (upper_bounds[k-1] - lower_bounds[k-1]) / (grid_points - 1)

        # the conditional means at each grid point (vector of length
        # grid points)
        cond_means = means[k] + cov_pre_curr * (z_grid - means[k-1])
        
        fut_arg = np.zeros(grid_points)
        eff_arg = np.zeros(grid_points)
        for i in range(grid_points):
            fut_arg[i] = std_norm_cdf((lower_bounds[k] - cond_means[i]) / cond_scale) * density[i]
            eff_arg[i] = (1.0 - std_norm_cdf((upper_bounds[k] - cond_means[i]) / cond_scale)) * density[i]
            
        # Composite Trapezoidal Integration rule
        fut_probs[k] = (np.sum(fut_arg) - 0.5 * (fut_arg[0] + fut_arg[-1])) * dz
        eff_probs[k] = (np.sum(eff_arg) - 0.5 * (eff_arg[0] + eff_arg[-1])) * dz
        
        # Grid transition to next stage
        if k < n_analyses - 1:
            next_lower, next_upper = lower_bounds[k], upper_bounds[k]
            
            if next_lower >= next_upper:
                print("Error: Upper bounds are lower than lower bounds.")
                return
            
            next_z_grid = np.linspace(next_lower, next_upper, grid_points)

            # the recursive density profile is defined by the 
            # Chapman-Kolmogorov-style integral
            # transition density matrix (transition_pdf) 
            # mapping a state y at stage k−1 to a state z at stage k
            transition_pdf = np.zeros((grid_points, grid_points))
            for r in range(grid_points):
                for c in range(grid_points):
                    diff = (next_z_grid[r] - cond_means[c]) / cond_scale
                    transition_pdf[r, c] = std_norm_pdf(diff) / cond_scale
            
            next_matrix = transition_pdf * density
            density = (np.sum(next_matrix, axis=1) - 0.5 * (next_matrix[:, 0] + next_matrix[:, -1])) * dz
            z_grid = next_z_grid
            
    return fut_probs, eff_probs


# high-level Python caller wrapper
def group_sequential_designs_numba(
        n_analyses=3,
        upper_bounds=[2.5, 2, 1.5],
        lower_bounds=[0, 0.75, 1.5],
        n_patients=20,
        null_hypothesis=0,
        alt_hypothesis=0.5,
        variance=1,
        grid_points=100):

    upper_bounds = np.asarray(upper_bounds, dtype=float)
    lower_bounds = np.asarray(lower_bounds, dtype=float)
    n_patients_analysis = n_patients * np.arange(1, n_analyses + 1)

    # standardize the null and alternative hypotheses
    # sqrt(information) * theta is the Wald statistic at each stage (Z_j)
    scale = np.sqrt(n_patients_analysis / (2 * variance))
    mean_0 = null_hypothesis * scale
    mean_1 = alt_hypothesis * scale

    futility_null, efficacy_null = _numba_grid_engine(
        n_analyses, upper_bounds, lower_bounds, n_patients_analysis, mean_0, grid_points
    )
    
    futility_alt, efficacy_alt = _numba_grid_engine(
        n_analyses, upper_bounds, lower_bounds, n_patients_analysis, mean_1, grid_points
    )

    alpha = efficacy_null.sum()
    power = efficacy_alt.sum()
    expected_sample_size = np.sum((futility_alt + efficacy_alt) * n_patients_analysis)

    return alpha, power, expected_sample_size

In [5]:
group_sequential_designs_mazin(null_hypothesis=1, alt_hypothesis=2, variance = 4)

(0.8517255510610606, 0.9991299311602759, 25.18292346656813)

In [6]:
group_sequential_designs_recursive(null_hypothesis=1, alt_hypothesis=2,variance = 4)

(0.8516884351704618, 0.9991502543410972, 25.18374765775883)

In [7]:
group_sequential_designs_numba(null_hypothesis=1, alt_hypothesis=2,variance = 4)

(0.8517047238251587, 0.9991412831853781, 25.18338380180087)

In [8]:
import timeit

runs = 1000
# Benchmark with 4 interim analyses
t_scipy = timeit.timeit('group_sequential_designs_mazin()', number=runs, globals=globals())
t_grid = timeit.timeit('group_sequential_designs_recursive()', number=runs, globals=globals())
t_numba = timeit.timeit('group_sequential_designs_numba()', number=runs, globals=globals())

print(f"SciPy Total Time ({runs} runs): {t_scipy:.4f} seconds")
print(f"1D Grid Total Time ({runs} runs): {t_grid:.4f} seconds")
print(f"Numba Total Time ({runs} runs): {t_numba:.4f} seconds")
print(f"Speedup factor v 1D: {t_scipy / t_grid:.1f}x faster")
print(f"Speedup factor v Numba: {t_scipy / t_numba:.1f}x faster")

SciPy Total Time (1000 runs): 2.0652 seconds
1D Grid Total Time (1000 runs): 0.4923 seconds
Numba Total Time (1000 runs): 0.0697 seconds
Speedup factor v 1D: 4.2x faster
Speedup factor v Numba: 29.6x faster
